# Highlight

This notebook shows a short example of how to process the CORD-19 data and 
introduces how we manage the full text indexing, if available. Later, it shows
how to query and highlight the words in the text that match the query.

In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import pandas as pd
import lucene

# Import the classes that we develop to process the data
from src.retrieval.index_reader import Reader
from src.index_writer import Indexer
from src.CordReader import CordReader

In [3]:
lucene.initVM(vmargs=["-Djava.awt.headless=true"])

Dec 05, 2025 3:02:48 AM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled


## Write some sample data in cord19 format

In [4]:
cord_19_path = "../sample_data/small_cord_19.csv"
index_path = "/tmp/cord_test"

# We read the three entries from our CSV file
df = pd.read_csv(cord_19_path)
df.head()

,cord_uid,source_x,title,abstract,publish_time,journal,authors,url
0,ug7v899j,PMC,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...,2001-07-04,BMC Infect Dis,"Madani, Tariq A; Al-Ghamdi, Aisha A",https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3...
1,02tnwd4m,PMC,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...,2000-08-15,Respir Res,"Vliet, Albert van der; Eiserich, Jason P; Cros...",https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...
2,ejv2xln0,PMC,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...,2000-08-25,Respir Res,"Crouch, Erika C",https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...


Notice that the columns in this CSV are missing some of the expected values
in the CORD-19 lucene schema, so we fake them for the sake of the example.

In [5]:
df["doc_id"] = df.cord_uid
df["source"] = "test"
df["pub_date"] = "2025-10-10"
df["modalities"] = "oth"
df["num_figures"] = 2

# pmcid is an important field because we use them in the CordReader class to be
# able to map an entry from a document metadata to the full text that is stored
# in another json file. Adding the full text to the dataframe would not be convenient
# given the format used by CORD-19 of several sentence blocks.
df["pmcid"] = [
    "ug7v899j",
    "02tnwd4m",
    "ejv2xln0",
]

# The captions entry is an array of objects that map the figure number in the paper
# with the caption text
captions = [
    [{"figure_id": 1, "text": "test"}],
    [{"figure_id": 1, "text": "test"}],
    [{"figure_id": 1, "text": "test"}],
]
df["captions"] = captions
df["otherid"] = df.cord_uid

writer = Indexer(store_path=index_path, create_mode=True)
# In our wrapper, we use a dataframe to provide the metadata and the CordReader
# parses the full_text, if existent. In this example, the full text is in
# the `sample_fulltext` folder where `pmcid_2_fulltext` is an index to map
# a pmcid to the document with the fulltext, and each other json file contains
# a sample full text for the 3 documents in the sample collection
writer.index_from_dataframe(df, ft_provider=CordReader("./sample_fulltext"))

full text mapping found


## Read from generated indexes

In [6]:
index_path = "/tmp/cord_test"
reader = Reader(index_path)

In [7]:
query = "respiratory"
# modalities = ['oth']
results = reader.search(
    terms=query,
    start_date=None,
    end_date=None,
    modalities=None,
    only_with_images=True,
    ft=True,
    highlight=True,
)

In [8]:
from org.apache.lucene.analysis.standard import StandardAnalyzer
from org.apache.lucene.search.highlight import (
    SimpleHTMLFormatter,
    QueryScorer,
    Highlighter,
    SimpleSpanFragmenter,
    GradientFormatter,
)
from java.io import StringReader
from IPython.display import Markdown as md

In [9]:
highlight_formatter = SimpleHTMLFormatter()
# highlight_formatter = GradientFormatter(10.0, '#000000', '#000000', '#FFFFFF', '#FF0000')
query_scorer = QueryScorer(reader.get_last_query())
highlighter = Highlighter(highlight_formatter, query_scorer)
analyzer = StandardAnalyzer()

In [11]:
# See that the results include html tags for the highlights
results

[SearchResult(id=None, title='Nitric oxide: a pro-inflammatory mediator in lung disease?', abstract='Inflammatory diseases of the <B>respiratory</B> tract are commonly associated with elevated production of nitric oxide (NO•) and increased indices of NO• -dependent oxidative stress. Although NO• is known to have anti-microbial, anti-inflammatory and anti-oxidant properties, various lines of evidence support the contribution of NO• to lung injury in several disease models. On the basis of biochemical... myeloperoxidase and eosinophil peroxidase might be operative during conditions of inflammation. Because of the overwhelming literature on NO• generation and activities in the <B>respiratory</B> tract, it would', publish_date='2025-10-10', modalities=['oth'], num_figures=2, url='https://www.ncbi.nlm.nih.gov/pmc/articles/PMC59543/', full_text='', journal='Respir Res', authors='Vliet, Albert van der; Eiserich, Jason P; Cross, Carroll E', captions=[], modalities_count={}),
 SearchResult(id=N

In [12]:
fragmenter = SimpleSpanFragmenter(query_scorer, 200)
highlighter.setTextFragmenter(fragmenter)

# abstract = ",".join(results[0].abstract)
abstract = results[0].abstract
title = results[0].title

ts_abs = analyzer.tokenStream("abstract", StringReader(abstract))
frag_abs = highlighter.getBestFragments(ts_abs, abstract, 3, "...")

ts_title = analyzer.tokenStream("title", StringReader(title))
frag_title = highlighter.getBestFragments(ts_title, title, 3, "...")

ts_mods = analyzer.tokenStream(
    "modalities", StringReader("".join(results[0].modalities))
)
frag_mods = highlighter.getBestFragments(
    ts_mods, "".join(results[0].modalities), 3, "..."
)

In [13]:
md(f"{frag_abs}")

Inflammatory diseases of the <B><B>respiratory</B></B> tract are commonly associated with elevated production of nitric oxide (NO•) and increased indices of NO• -dependent oxidative stress. Although NO• is known to have anti-microbial, anti-inflammatory and anti-oxidant properties, various lines of evidence support the contribution of NO• to lung injury in several disease models. On the basis of biochemical... myeloperoxidase and eosinophil peroxidase might be operative during conditions of inflammation. Because of the overwhelming literature on NO• generation and activities in the <B><B>respiratory</B></B> tract, it would

In [18]:
# no highlight found in title
md(f"{frag_title}")

In [19]:
results[0]

SearchResult(id=None, title='Nitric oxide: a pro-inflammatory mediator in lung disease?', abstract='Inflammatory diseases of the <B>respiratory</B> tract are commonly associated with elevated production of nitric oxide (NO•) and increased indices of NO• -dependent oxidative stress. Although NO• is known to have anti-microbial, anti-inflammatory and anti-oxidant properties, various lines of evidence support the contribution of NO• to lung injury in several disease models. On the basis of biochemical... myeloperoxidase and eosinophil peroxidase might be operative during conditions of inflammation. Because of the overwhelming literature on NO• generation and activities in the <B>respiratory</B> tract, it would', publish_date='2025-10-10', modalities=['oth'], num_figures=2, url='https://www.ncbi.nlm.nih.gov/pmc/articles/PMC59543/', full_text='', journal='Respir Res', authors='Vliet, Albert van der; Eiserich, Jason P; Cross, Carroll E', captions=[], modalities_count={})